# Fase 2 — Entendimento dos Dados
Objetivo: descobrir problemas nas bases. Nenhuma correção é feita aqui.

In [1]:
import pandas as pd
from pathlib import Path

SEED = 42
RAIZ = Path('..')
DADOS_BRUTOS = RAIZ / 'dados' / 'brutos'
ESTADO = RAIZ / 'estado'
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

## 1. Carga das bases

In [2]:
# feriado_estadual mistura 0 numérico com letras; lido como texto para não perder a informação
lojas = pd.read_csv(DADOS_BRUTOS / 'loja.csv')
vendas = pd.read_csv(DADOS_BRUTOS / 'vendas.csv', parse_dates=['data'], dtype={'feriado_estadual': str})

print(f'lojas : {lojas.shape[0]:>9,} linhas x {lojas.shape[1]} colunas')
print(f'vendas: {vendas.shape[0]:>9,} linhas x {vendas.shape[1]} colunas')
print(f"período de vendas: {vendas['data'].min().date()} a {vendas['data'].max().date()} "
      f"({vendas['data'].nunique()} datas distintas)")
print('tipo da coluna data:', vendas['data'].dtype)

lojas :     1,115 linhas x 5 colunas
vendas: 1,017,209 linhas x 9 colunas
período de vendas: 2013-01-01 a 2015-07-31 (942 datas distintas)
tipo da coluna data: datetime64[ns]


## 2. Primeiras linhas

In [3]:
lojas.head()

,loja,tipo_loja,sortimento,distancia_concorrente,promo_continua
0,1,c,a,1270.0,0
1,2,a,a,570.0,1
2,3,a,a,14130.0,1
3,4,c,c,620.0,0
4,5,a,a,29910.0,0


In [4]:
vendas.head()

,loja,dia_semana,data,faturamento,clientes,aberta,promo,feriado_estadual,feriado_escolar
0,1,5,2015-07-31,5263,555,1,1,0,1
1,2,5,2015-07-31,6064,625,1,1,0,1
2,3,5,2015-07-31,8314,821,1,1,0,1
3,4,5,2015-07-31,13995,1498,1,1,0,1
4,5,5,2015-07-31,4822,559,1,1,0,1


## 3. Evidências para o dicionário de dados

In [5]:
def resumo_colunas(df):
    return pd.DataFrame({
        'tipo': df.dtypes.astype(str),
        'valores_distintos': df.nunique(),
        'minimo': df.min(numeric_only=False),
        'maximo': df.max(numeric_only=False),
        'exemplos': [df[col].dropna().unique()[:6].tolist() for col in df.columns],
    })

print('LOJAS'); display(resumo_colunas(lojas))
print('VENDAS'); display(resumo_colunas(vendas))

LOJAS


,tipo,valores_distintos,minimo,maximo,exemplos
loja,int64,1115,1,1115,"[1, 2, 3, 4, 5, 6]"
tipo_loja,object,4,a,d,"[c, a, d, b]"
sortimento,object,3,a,c,"[a, c, b]"
distancia_concorrente,float64,654,20.0,75860.0,"[1270.0, 570.0, 14130.0, 620.0, 29910.0, 310.0]"
promo_continua,int64,2,0,1,"[0, 1]"


VENDAS


,tipo,valores_distintos,minimo,maximo,exemplos
loja,int64,1115,1,1115,"[1, 2, 3, 4, 5, 6]"
dia_semana,int64,7,1,7,"[5, 4, 3, 2, 1, 7]"
data,datetime64[ns],942,2013-01-01 00:00:00,2015-07-31 00:00:00,"[2015-07-31 00:00:00, 2015-07-30 00:00:00, 201..."
faturamento,int64,21734,0,41551,"[5263, 6064, 8314, 13995, 4822, 5651]"
clientes,int64,4086,0,7388,"[555, 625, 821, 1498, 559, 589]"
aberta,int64,2,0,1,"[1, 0]"
promo,int64,2,0,1,"[1, 0]"
feriado_estadual,object,4,0,c,"[0, a, b, c]"
feriado_escolar,int64,2,0,1,"[1, 0]"


In [6]:
# dia_semana: qual número corresponde a qual dia do calendário
mapa_dia = (vendas.assign(dia_calendario=vendas['data'].dt.day_name())
            .groupby('dia_semana')['dia_calendario'].agg(lambda s: s.value_counts().to_dict()))
print(mapa_dia.to_string())

dia_semana
1       {'Monday': 144730}
2      {'Tuesday': 145664}
3    {'Wednesday': 145665}
4     {'Thursday': 145845}
5       {'Friday': 145845}
6     {'Saturday': 144730}
7       {'Sunday': 144730}


In [7]:
print('valores de feriado_estadual:'); print(vendas['feriado_estadual'].value_counts().to_string())
print('\nvalores de feriado_escolar:'); print(vendas['feriado_escolar'].value_counts().to_string())

valores de feriado_estadual:
feriado_estadual
0    986159
a     20260
b      6690
c      4100

valores de feriado_escolar:
feriado_escolar
0    835488
1    181721


## 4. Colunas com nomes parecidos

In [8]:
# promo (diária, em vendas) x promo_continua (fixa por loja, em lojas)
promo_por_loja = vendas.groupby('loja')['promo'].mean().rename('pct_dias_promo')
comparacao = lojas.set_index('loja')[['promo_continua']].join(promo_por_loja)
print('% de dias com promo, por grupo de promo_continua:')
print(comparacao.groupby('promo_continua')['pct_dias_promo'].describe().round(3).to_string())

# promo_continua varia dentro da loja? (se sim, não seria atributo fixo)
print('\nlojas com mais de um valor de promo_continua:', lojas.groupby('loja')['promo_continua'].nunique().gt(1).sum())

% de dias com promo, por grupo de promo_continua:
                count   mean    std    min    25%    50%    75%    max
promo_continua                                                        
0               544.0  0.382  0.001  0.377  0.382  0.382  0.382  0.383
1               571.0  0.381  0.002  0.377  0.377  0.382  0.382  0.382

lojas com mais de um valor de promo_continua: 0


In [9]:
# feriado_estadual x feriado_escolar: são independentes ou um implica o outro?
print(pd.crosstab(vendas['feriado_estadual'], vendas['feriado_escolar'], margins=True).to_string())

feriado_escolar        0       1      All
feriado_estadual                         
0                 819949  166210   986159
a                  15055    5205    20260
b                    484    6206     6690
c                      0    4100     4100
All               835488  181721  1017209


## 5. Nulos e duplicatas

In [10]:
print('NULOS - lojas'); print(lojas.isna().sum().to_string())
print('\nNULOS - vendas'); print(vendas.isna().sum().to_string())

print('\nlojas com distancia_concorrente nula:')
display(lojas[lojas['distancia_concorrente'].isna()])

NULOS - lojas
loja                     0
tipo_loja                0
sortimento               0
distancia_concorrente    3
promo_continua           0

NULOS - vendas
loja                0
dia_semana          0
data                0
faturamento         0
clientes            0
aberta              0
promo               0
feriado_estadual    0
feriado_escolar     0

lojas com distancia_concorrente nula:


,loja,tipo_loja,sortimento,distancia_concorrente,promo_continua
290,291,d,a,NaN,0
621,622,a,c,NaN,0
878,879,d,a,NaN,1


In [11]:
print('DUPLICATAS')
print('lojas  - linhas idênticas      :', lojas.duplicated().sum())
print('lojas  - chave loja repetida   :', lojas['loja'].duplicated().sum())
print('vendas - linhas idênticas      :', vendas.duplicated().sum())
print('vendas - chave (loja, data)    :', vendas.duplicated(['loja', 'data']).sum())

print('\nlojas no cadastro sem vendas :', len(set(lojas['loja']) - set(vendas['loja'])))
print('lojas em vendas sem cadastro  :', len(set(vendas['loja']) - set(lojas['loja'])))

DUPLICATAS
lojas  - linhas idênticas      : 0
lojas  - chave loja repetida   : 0
vendas - linhas idênticas      : 0
vendas - chave (loja, data)    : 0



lojas no cadastro sem vendas : 0


lojas em vendas sem cadastro  : 0


In [12]:
# estado salvo para os próximos passos não recarregarem os CSVs
ESTADO.mkdir(exist_ok=True)
lojas.to_pickle(ESTADO / 'lojas.pkl')
vendas.to_pickle(ESTADO / 'vendas.pkl')

## 6. Faturamento x abertura

In [2]:
resumo_aberta = vendas.groupby('aberta')['faturamento'].agg(contagem='count', media='mean', mediana='median')
print(resumo_aberta.round(1).to_string())

        contagem   media  mediana
aberta                           
0         172817     0.0      0.0
1         844392  6955.5   6369.0


## 7. Incoerências entre aberta, faturamento e clientes

In [3]:
checagens = {
    'fechada com faturamento > 0': (vendas['aberta'] == 0) & (vendas['faturamento'] > 0),
    'fechada com clientes > 0':    (vendas['aberta'] == 0) & (vendas['clientes'] > 0),
    'aberta com faturamento = 0':  (vendas['aberta'] == 1) & (vendas['faturamento'] == 0),
    'aberta com clientes = 0':     (vendas['aberta'] == 1) & (vendas['clientes'] == 0),
    'faturamento > 0 e clientes = 0': (vendas['faturamento'] > 0) & (vendas['clientes'] == 0),
}
for nome, filtro in checagens.items():
    print(f'{nome:<32}: {filtro.sum()}')

fechada com faturamento > 0     : 0
fechada com clientes > 0        : 0
aberta com faturamento = 0      : 54
aberta com clientes = 0         : 52
faturamento > 0 e clientes = 0  : 0


In [4]:
aberta_sem_venda = vendas[checagens['aberta com faturamento = 0']]
print('clientes nessas linhas:', aberta_sem_venda['clientes'].value_counts().to_dict())
print('lojas distintas:', aberta_sem_venda['loja'].nunique())
print('ocorrências por loja (top 5):', aberta_sem_venda['loja'].value_counts().head().to_dict())
print('por dia_semana:', aberta_sem_venda['dia_semana'].value_counts().sort_index().to_dict())
print('com promo:', aberta_sem_venda['promo'].sum(), '| com feriado estadual:', (aberta_sem_venda['feriado_estadual'] != '0').sum())
print('datas distintas:', aberta_sem_venda['data'].nunique())
print('datas mais frequentes:', aberta_sem_venda['data'].dt.date.value_counts().head(3).to_dict())

clientes nessas linhas: {0: 52, 3: 1, 5: 1}
lojas distintas: 41
ocorrências por loja (top 5): {28: 3, 835: 2, 1039: 2, 665: 2, 623: 2}
por dia_semana: {1: 3, 2: 6, 3: 14, 4: 18, 5: 7, 6: 6}
com promo: 21 | com feriado estadual: 0
datas distintas: 49
datas mais frequentes: {datetime.date(2014, 7, 24): 4, datetime.date(2014, 7, 23): 2, datetime.date(2014, 9, 11): 2}


## 8. Proporção de domingos abertos por loja

In [5]:
domingos = vendas[vendas['dia_semana'] == 7]
prop_domingo_aberta = domingos.groupby('loja')['aberta'].mean().rename('prop_domingo_aberta')

print('lojas que nunca abriram domingo :', (prop_domingo_aberta == 0).sum())
print('lojas que abriram algum domingo :', (prop_domingo_aberta > 0).sum())
faixas = pd.cut(prop_domingo_aberta, bins=[-0.001, 0, 0.1, 0.5, 0.9, 1.0],
                labels=['0%', '(0-10%]', '(10-50%]', '(50-90%]', '(90-100%]'])
print('\ndistribuição:'); print(faixas.value_counts().sort_index().to_string())
print('\nlojas que abrem domingo - describe:')
print(prop_domingo_aberta[prop_domingo_aberta > 0].describe().round(3).to_string())

lojas que nunca abriram domingo : 1082
lojas que abriram algum domingo : 33

distribuição:
prop_domingo_aberta
0%           1082
(0-10%]         0
(10-50%]        1
(50-90%]       16
(90-100%]      16

lojas que abrem domingo - describe:
count    33.000
mean      0.818
std       0.202
min       0.164
25%       0.657
50%       0.806
75%       1.000
max       1.000


## 9. Venda média: todos os dias x só dias abertos

In [6]:
venda_media = pd.DataFrame({
    'media_todos_dias': vendas.groupby('loja')['faturamento'].mean(),
    'media_dias_abertos': vendas[vendas['aberta'] == 1].groupby('loja')['faturamento'].mean(),
    'pct_dias_abertos': vendas.groupby('loja')['aberta'].mean(),
}).join(prop_domingo_aberta)
venda_media['abre_domingo'] = venda_media['prop_domingo_aberta'] > 0
venda_media['diferenca_pct'] = (venda_media['media_dias_abertos'] / venda_media['media_todos_dias'] - 1) * 100

comparacao_domingo = venda_media.groupby('abre_domingo').agg(
    lojas=('media_todos_dias', 'size'),
    media_todos_dias=('media_todos_dias', 'mean'),
    media_dias_abertos=('media_dias_abertos', 'mean'),
    diferenca_pct_mediana=('diferenca_pct', 'median'),
    pct_dias_abertos_medio=('pct_dias_abertos', 'mean'),
)
print(comparacao_domingo.round(2).to_string())

              lojas  media_todos_dias  media_dias_abertos  diferenca_pct_mediana  pct_dias_abertos_medio
abre_domingo                                                                                            
False          1082           5692.62             6891.44                  20.92                    0.83
True             33           8081.56             8336.47                   3.97                    0.96


In [7]:
# a média sobre todos os dias é a média dos dias abertos multiplicada pela fração de dias abertos?
reconstruida = venda_media['media_dias_abertos'] * venda_media['pct_dias_abertos']
erro = (reconstruida - venda_media['media_todos_dias']).abs()
print('erro máximo da identidade (R$):', round(erro.max(), 6))
print('correlação media_todos_dias x pct_dias_abertos :', round(venda_media['media_todos_dias'].corr(venda_media['pct_dias_abertos']), 3))
print('correlação media_dias_abertos x pct_dias_abertos:', round(venda_media['media_dias_abertos'].corr(venda_media['pct_dias_abertos']), 3))

# posição no ranking muda conforme a métrica?
rank_todos = venda_media['media_todos_dias'].rank(ascending=False)
rank_abertos = venda_media['media_dias_abertos'].rank(ascending=False)
salto = (rank_todos - rank_abertos)
print('\nmudança de posição no ranking (todos x abertos), por grupo:')
print(salto.groupby(venda_media['abre_domingo']).describe().round(1).to_string())

erro máximo da identidade (R$): 0.0
correlação media_todos_dias x pct_dias_abertos : 0.259
correlação media_dias_abertos x pct_dias_abertos: 0.153

mudança de posição no ranking (todos x abertos), por grupo:
               count   mean   std    min    25%    50%   75%    max
abre_domingo                                                       
False         1082.0    3.3   9.7  -14.0   -1.8    2.0   6.0  134.0
True            33.0 -108.4  77.4 -238.0 -162.0 -122.0 -35.0   38.0


In [8]:
venda_media.to_pickle(ESTADO / 'venda_media.pkl')
prop_domingo_aberta.to_pickle(ESTADO / 'prop_domingo_aberta.pkl')

## 10. Completude do histórico por loja

In [2]:
dias_por_loja = vendas.groupby('loja').size().rename('dias_registrados')
print('distribuição da contagem de dias por loja:')
print(dias_por_loja.value_counts().rename_axis('dias_registrados').rename('lojas').to_string())

distribuição da contagem de dias por loja:
dias_registrados
942    934
758    180
941      1


In [3]:
calendario = pd.DatetimeIndex(sorted(vendas['data'].unique()))
print('datas no calendário completo:', len(calendario), '| esperado de 01/01/2013 a 31/07/2015:',
      len(pd.date_range('2013-01-01', '2015-07-31')))

def datas_faltantes(loja):
    return calendario.difference(pd.DatetimeIndex(vendas.loc[vendas['loja'] == loja, 'data']))

lojas_incompletas = dias_por_loja[dias_por_loja == 758].index
loja_exemplo = lojas_incompletas[0]
faltantes = datas_faltantes(loja_exemplo)
print(f'\nloja {loja_exemplo}: {len(faltantes)} datas faltando, de {faltantes.min().date()} a {faltantes.max().date()}')
print('intervalo contínuo?', len(faltantes) == len(pd.date_range(faltantes.min(), faltantes.max())))

loja_941 = dias_por_loja[dias_por_loja == 941].index
for loja in loja_941:
    print(f'loja {loja} (941 dias): falta {[d.date() for d in datas_faltantes(loja)]}')

datas no calendário completo: 942 | esperado de 01/01/2013 a 31/07/2015: 942

loja 13: 184 datas faltando, de 2014-07-01 a 2014-12-31
intervalo contínuo? True
loja 988 (941 dias): falta [datetime.date(2013, 1, 1)]


## 11. Como os dias faltantes aparecem

In [4]:
no_buraco = vendas[(vendas['loja'] == loja_exemplo) & vendas['data'].between(faltantes.min(), faltantes.max())]
print(f'linhas da loja {loja_exemplo} dentro do intervalo faltante:', len(no_buraco))
print('nulos na base inteira:', int(vendas.isna().sum().sum()))
vizinhos = vendas[(vendas['loja'] == loja_exemplo) &
                  vendas['data'].between(faltantes.min() - pd.Timedelta(days=2), faltantes.max() + pd.Timedelta(days=2))]
print('\nregistros nas bordas do buraco:')
print(vizinhos.sort_values('data')[['loja', 'data', 'faturamento', 'aberta']].to_string(index=False))

linhas da loja 13 dentro do intervalo faltante: 0
nulos na base inteira: 0

registros nas bordas do buraco:
 loja       data  faturamento  aberta
   13 2014-06-29            0       0
   13 2014-06-30         9192       1
   13 2015-01-01            0       0
   13 2015-01-02         4348       1


## 12. O buraco é o mesmo em todas as lojas incompletas?

In [5]:
buracos = {loja: tuple(datas_faltantes(loja)) for loja in lojas_incompletas}
buracos_distintos = pd.Series(buracos).map(lambda d: (d[0].date(), d[-1].date(), len(d))).value_counts()
print('lojas no grupo incompleto:', len(lojas_incompletas))
print('padrões de buraco distintos (início, fim, dias):')
print(buracos_distintos.to_string())

# todas as outras lojas estão presentes nesse período?
periodo_buraco = vendas['data'].between(faltantes.min(), faltantes.max())
print('\nlojas com registro dentro do período:', vendas.loc[periodo_buraco, 'loja'].nunique())

lojas no grupo incompleto: 180
padrões de buraco distintos (início, fim, dias):
(2014-07-01, 2014-12-31, 184)    180

lojas com registro dentro do período: 935


## 13. Impacto no ranking: soma x média em dia aberto

In [6]:
from math import comb

ranking = pd.DataFrame({
    'soma_faturamento': vendas.groupby('loja')['faturamento'].sum(),
    'media_dias_abertos': venda_media['media_dias_abertos'],
})
ranking['incompleta'] = ranking.index.isin(lojas_incompletas)

def prob_ao_menos(k, total, sucessos_pop, sorteados):
    # hipergeométrica: chance de k ou mais incompletas no pior quartil se a distribuição fosse aleatória
    return sum(comb(sucessos_pop, i) * comb(total - sucessos_pop, sorteados - i)
               for i in range(k, min(sucessos_pop, sorteados) + 1)) / comb(total, sorteados)

n_incompletas = ranking['incompleta'].sum()
linhas = []
for metrica in ['soma_faturamento', 'media_dias_abertos']:
    pior_quartil = ranking[metrica] <= ranking[metrica].quantile(0.25)
    no_pior = (pior_quartil & ranking['incompleta']).sum()
    esperado = n_incompletas * pior_quartil.mean()
    linhas.append({
        'metrica': metrica,
        'lojas_no_pior_quartil': pior_quartil.sum(),
        'incompletas_no_pior_quartil': no_pior,
        'pct_das_incompletas': round(no_pior / n_incompletas * 100, 1),
        'esperado_por_acaso': round(esperado, 1),
        'p_valor': prob_ao_menos(no_pior, len(ranking), pior_quartil.sum(), n_incompletas),
    })
print(f'lojas incompletas: {n_incompletas} de {len(ranking)}')
print(pd.DataFrame(linhas).to_string(index=False))

lojas incompletas: 180 de 1115
           metrica  lojas_no_pior_quartil  incompletas_no_pior_quartil  pct_das_incompletas  esperado_por_acaso      p_valor
  soma_faturamento                    279                           98                 54.4                45.0 8.562967e-21
media_dias_abertos                    279                           45                 25.0                45.0 5.362384e-01


In [7]:
dias_por_loja.to_pickle(ESTADO / 'dias_por_loja.pkl')
pd.Series(lojas_incompletas).to_pickle(ESTADO / 'lojas_incompletas.pkl')

## 14. Base de lojas — os 3 nulos de distancia_concorrente

In [2]:
perfil_lojas = lojas.set_index('loja').join(venda_media[['media_dias_abertos', 'pct_dias_abertos', 'prop_domingo_aberta']]).join(dias_por_loja)
perfil_lojas['clientes_medios'] = vendas[vendas['aberta'] == 1].groupby('loja')['clientes'].mean()
perfil_lojas['pct_venda_rede'] = perfil_lojas['media_dias_abertos'].rank(pct=True).round(2)

nulas = perfil_lojas[perfil_lojas['distancia_concorrente'].isna()]
print(nulas.round(2).to_string())

     tipo_loja sortimento  distancia_concorrente  promo_continua  media_dias_abertos  pct_dias_abertos  prop_domingo_aberta  dias_registrados  clientes_medios  pct_venda_rede
loja                                                                                                                                                                          
291          d          a                    NaN               0             8023.04              0.83                  0.0               942           840.35            0.76
622          a          c                    NaN               0             4317.96              0.83                  0.0               942           430.00            0.08
879          d          a                    NaN               1             3762.98              0.82                  0.0               758           347.33            0.04


In [3]:
# se o nulo significasse "sem concorrente", o esperado seria parecer com as lojas de concorrente mais distante
perfil_lojas['faixa_distancia'] = pd.qcut(perfil_lojas['distancia_concorrente'], 4, labels=['Q1 perto', 'Q2', 'Q3', 'Q4 longe'])
por_faixa = perfil_lojas.groupby('faixa_distancia', observed=True).agg(
    lojas=('media_dias_abertos', 'size'),
    distancia_mediana=('distancia_concorrente', 'median'),
    venda_mediana=('media_dias_abertos', 'median'),
    clientes_medianos=('clientes_medios', 'median'),
)
print(por_faixa.round(0).to_string())
print('\ncorrelação de Spearman distância x venda média em dia aberto:',
      round(perfil_lojas['distancia_concorrente'].corr(perfil_lojas['media_dias_abertos'], method='spearman'), 3))
print('valores mais frequentes de distância (checando código especial tipo 99999):')
print(lojas['distancia_concorrente'].value_counts().head(5).to_string())

                 lojas  distancia_mediana  venda_mediana  clientes_medianos
faixa_distancia                                                            
Q1 perto           278              300.0         6707.0              804.0
Q2                 278             1420.0         6358.0              656.0
Q3                 278             3855.0         6486.0              639.0
Q4 longe           278            13595.0         6687.0              635.0



correlação de Spearman distância x venda média em dia aberto: -0.028
valores mais frequentes de distância (checando código especial tipo 99999):
distancia_concorrente
250.0     12
1200.0     9
50.0       8
350.0      8
190.0      8


## 15. Estatísticas descritivas do cadastro

In [4]:
estatisticas = lojas.drop(columns='loja').describe().T
estatisticas = estatisticas.rename(columns={'count': 'contagem', 'mean': 'media', 'std': 'desvio', 'min': 'minimo',
                                            '25%': 'q1', '50%': 'mediana', '75%': 'q3', 'max': 'maximo'})
print(estatisticas.round(2).to_string())
print('\ncategóricas:')
for col in ['tipo_loja', 'sortimento']:
    print(col, lojas[col].value_counts().to_dict())

                       contagem    media   desvio  minimo     q1  mediana      q3   maximo
distancia_concorrente    1112.0  5404.90  7663.17    20.0  717.5   2325.0  6882.5  75860.0
promo_continua           1115.0     0.51     0.50     0.0    0.0      1.0     1.0      1.0

categóricas:
tipo_loja {'a': 602, 'd': 348, 'c': 148, 'b': 17}
sortimento {'a': 593, 'c': 513, 'b': 9}


## 16. Assimetria e outliers de distancia_concorrente

In [5]:
import numpy as np
distancia = lojas['distancia_concorrente'].dropna()
q1, q3 = distancia.quantile([0.25, 0.75])
limite_superior = q3 + 1.5 * (q3 - q1)
print(f'assimetria (skew): {distancia.skew():.2f} | curtose: {distancia.kurt():.2f}')
print(f'média / mediana: {distancia.mean() / distancia.median():.2f}')
print(f'outliers pela regra do IQR (> {limite_superior:,.0f}): {(distancia > limite_superior).sum()} lojas ({(distancia > limite_superior).mean():.1%})')
print('percentis:', distancia.quantile([0.5, 0.9, 0.95, 0.99, 1.0]).round(0).to_dict())

assimetria (skew): 2.93 | curtose: 13.10
média / mediana: 2.32
outliers pela regra do IQR (> 16,130): 108 lojas (9.7%)
percentis: {0.5: 2325.0, 0.9: 15679.0, 0.95: 20145.0, 0.99: 35145.0, 1.0: 75860.0}


In [6]:
# como a distância fica depois da padronização que o K-Means exige
z = (distancia - distancia.mean()) / distancia.std()
print('escala padronizada (z-score) da distância bruta:')
print(z.quantile([0, 0.25, 0.5, 0.75, 0.95, 1.0]).round(2).to_dict())
print(f'metade das lojas espremida numa faixa de {z.quantile(0.75) - z.quantile(0.25):.2f} desvio; a mais distante fica a {z.max():.2f} desvios da média')

# peso das maiores distâncias na variância (é a variância que a padronização divide)
desvios2 = (distancia - distancia.mean()) ** 2
top5 = distancia >= distancia.quantile(0.95)
print(f'as 5% lojas mais distantes respondem por {desvios2[top5].sum() / desvios2.sum():.1%} da variância da coluna')

distancia_log = np.log1p(distancia)
z_log = (distancia_log - distancia_log.mean()) / distancia_log.std()
print(f'\ncomparação log1p: skew {distancia_log.skew():.2f} | z vai de {z_log.min():.2f} a {z_log.max():.2f}')

escala padronizada (z-score) da distância bruta:
{0.0: -0.7, 0.25: -0.61, 0.5: -0.4, 0.75: 0.19, 0.95: 1.92, 1.0: 9.19}
metade das lojas espremida numa faixa de 0.80 desvio; a mais distante fica a 9.19 desvios da média
as 5% lojas mais distantes respondem por 59.6% da variância da coluna

comparação log1p: skew -0.35 | z vai de -2.96 a 2.31


## 17. promo_continua tem relação com o comportamento comercial?

In [7]:
print(perfil_lojas.groupby('promo_continua')[['media_dias_abertos', 'clientes_medios', 'pct_dias_abertos']].median().round(2).to_string())
print('\ncruzamento com tipo_loja:')
print(pd.crosstab(lojas['tipo_loja'], lojas['promo_continua'], normalize='index').round(2).to_string())

                media_dias_abertos  clientes_medios  pct_dias_abertos
promo_continua                                                       
0                          6936.31           737.44              0.83
1                          6267.49           620.41              0.83

cruzamento com tipo_loja:
promo_continua     0     1
tipo_loja                 
a               0.52  0.48
b               0.71  0.29
c               0.49  0.51
d               0.42  0.58


In [8]:
perfil_lojas.to_pickle(ESTADO / 'perfil_lojas.pkl')

## 18. Do que são feitos tipo_loja e sortimento

In [2]:
print(pd.crosstab(lojas['tipo_loja'], lojas['sortimento'], margins=True, margins_name='total').to_string())
print('\n% de cada tipo por sortimento:')
print(pd.crosstab(lojas['tipo_loja'], lojas['sortimento'], normalize='index').round(2).to_string())

sortimento    a  b    c  total
tipo_loja                     
a           381  0  221    602
b             7  9    1     17
c            77  0   71    148
d           128  0  220    348
total       593  9  513   1115

% de cada tipo por sortimento:
sortimento     a     b     c
tipo_loja                   
a           0.63  0.00  0.37
b           0.41  0.53  0.06
c           0.52  0.00  0.48
d           0.37  0.00  0.63


## 19. Comportamento comercial por tipo_loja

In [3]:
dias_abertos = vendas[vendas['aberta'] == 1]
# ticket como razão de somas, para não quebrar nos dias com 0 cliente
ticket = dias_abertos.groupby('loja')['faturamento'].sum() / dias_abertos.groupby('loja')['clientes'].sum()
perfil_lojas['ticket_medio'] = ticket

por_tipo = perfil_lojas.groupby('tipo_loja').agg(
    lojas=('media_dias_abertos', 'size'),
    venda_media_dia_aberto=('media_dias_abertos', 'mean'),
    clientes_medios_dia=('clientes_medios', 'mean'),
    ticket_medio=('ticket_medio', 'mean'),
    prop_domingos_abertos=('prop_domingo_aberta', 'mean'),
    lojas_que_abrem_domingo=('prop_domingo_aberta', lambda s: (s > 0).sum()),
    distancia_mediana=('distancia_concorrente', 'median'),
)
rede = perfil_lojas.agg({'media_dias_abertos': 'mean', 'clientes_medios': 'mean', 'ticket_medio': 'mean', 'prop_domingo_aberta': 'mean'})
print(por_tipo.round(2).to_string())
print('\nrede:', rede.round(2).to_dict())

           lojas  venda_media_dia_aberto  clientes_medios_dia  ticket_medio  prop_domingos_abertos  lojas_que_abrem_domingo  distancia_mediana
tipo_loja                                                                                                                                     
a            602                 6912.54               792.13          8.96                   0.01                       13             1790.0
b             17                10110.78              1998.68          5.17                   0.98                       17              900.0
c            148                 6917.20               812.00          8.74                   0.00                        0             1660.0
d            348                 6823.74               604.21         11.43                   0.01                        3             5040.0

rede: {'media_dias_abertos': 6934.21, 'clientes_medios': 754.51, 'ticket_medio': 9.64, 'prop_domingo_aberta': 0.02}


In [4]:
# dispersão dentro de cada tipo: o quanto as lojas de um mesmo tipo são parecidas
dispersao = perfil_lojas.groupby('tipo_loja')[['media_dias_abertos', 'clientes_medios', 'ticket_medio']].agg(
    lambda s: s.std() / s.mean()).round(2)
dispersao.columns = [f'cv_{c}' for c in dispersao.columns]
print('coeficiente de variação dentro de cada tipo:')
print(dispersao.to_string())

coeficiente de variação dentro de cada tipo:
           cv_media_dias_abertos  cv_clientes_medios  cv_ticket_medio
tipo_loja                                                            
a                           0.37                0.43             0.16
b                           0.45                0.43             0.23
c                           0.31                0.35             0.16
d                           0.26                0.26             0.14


In [5]:
# tipo b em detalhe: o padrão vale para todas as 17 lojas ou é puxado por poucas?
colunas_b = ['sortimento', 'media_dias_abertos', 'clientes_medios', 'ticket_medio', 'prop_domingo_aberta', 'distancia_concorrente']
tipo_b = perfil_lojas[perfil_lojas['tipo_loja'] == 'b'][colunas_b].sort_values('clientes_medios')
print(tipo_b.round(2).to_string())
print('\nlojas b com ticket abaixo do menor ticket dos outros tipos:',
      (tipo_b['ticket_medio'] < perfil_lojas.loc[perfil_lojas['tipo_loja'] != 'b', 'ticket_medio'].min()).sum())
print('lojas b com clientes acima do p90 da rede:',
      (tipo_b['clientes_medios'] > perfil_lojas['clientes_medios'].quantile(0.9)).sum())

     sortimento  media_dias_abertos  clientes_medios  ticket_medio  prop_domingo_aberta  distancia_concorrente
loja                                                                                                          
1081          a             5924.92           937.18          6.32                 0.81                  400.0
274           b             4070.83           992.74          4.10                 0.99                 3640.0
85            a             7272.45          1015.40          7.16                 1.00                 1870.0
494           a             7649.30          1079.61          7.09                 1.00                 1260.0
512           b             5510.98          1287.97          4.28                 0.98                  590.0
353           b             5645.30          1449.53          3.89                 0.99                  900.0
676           b             7687.46          1687.59          4.56                 0.98                 1410.0
9

In [6]:
# onde o tipo b fica na escala padronizada que o K-Means vai enxergar
metricas = ['media_dias_abertos', 'clientes_medios', 'ticket_medio', 'prop_domingo_aberta']
z_metricas = (perfil_lojas[metricas] - perfil_lojas[metricas].mean()) / perfil_lojas[metricas].std()
print('z-score mediano por tipo:')
print(z_metricas.groupby(perfil_lojas['tipo_loja']).median().round(2).to_string())
print('\nparticipação do tipo b na rede:', f"{(perfil_lojas['tipo_loja'] == 'b').mean():.1%}")

z-score mediano por tipo:
           media_dias_abertos  clientes_medios  ticket_medio  prop_domingo_aberta
tipo_loja                                                                        
a                       -0.19            -0.11         -0.42                -0.17
b                        1.18             2.84         -2.38                 6.83
c                       -0.12             0.01         -0.60                -0.17
d                       -0.10            -0.48          0.92                -0.17

participação do tipo b na rede: 1.5%


In [7]:
perfil_lojas.to_pickle(ESTADO / 'perfil_lojas.pkl')